In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 1. 패키지

In [5]:
# python-dotenv, langchain, langchain-openai, langchain-pinecone, langchin-community, docx2txt
# langchain-upstage, langchain-ollama
# pandas

# 2. 환경변수 설정

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
UPSTAGE_API_KEY  = os.getenv("UPSTAGE_API_KEY")
OPENAI_LLM_MODEL = "gpt-4o-mini"
UPSTAGE_LLM_MODEL = "solar-pro4"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-large" # 차원수 3072
UPSTAGE_EMBEDDING_MODEL = "embedding-query" # 차원수 4092

PINECONE_INDEX_NAME = "better-rag-index"
PINECONE_INDEX_METRIC = 'cosine'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_DIMENSION = 3072

# 3. 문서를 chunk로 분할하기

In [15]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('tax_docx/소득세법_with_markdown.docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 1500글자 기준으로 분리
    chunk_overlap=100, 
    separators=['\n\n', '\n', ' ', '']
)
# documents = loader.load_and_split(text_splitter=text_splitter)
document = loader.load()
documents = text_splitter.split_documents(document)
print(f"총 {len(documents)}개 청크로 분리")

총 184개 청크로 분리


# 4. metadata 추가하기
- chunk 내용의 title, category, 조항

In [26]:
import re
def remove_special_chars(text:str) -> str:
    "특수문자 및 \n제거 (:, .는 그대로)"
    text = text.replace("\n", " ")
    # 한글, 영문, 숫자, 공백, . , : |-만 남기고 전부 제거
    cleaned = re.sub(r"[^가-힣a-zA-Z0-9\s\.,:|\-]", "", text).strip()
    # 불필요한 중복 공백을 제거
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned
content = " ▶️ **소득세 납세 의무 ** : \n\n|이 법 또는 「법인세법」에 따라 소득에 대한 소득세를 부과  "
remove_special_chars(content)

'소득세 납세 의무 : |이 법 또는 법인세법에 따라 소득에 대한 소득세를 부과'

In [27]:
remove_special_chars(documents[43].page_content)

'전문개정 2009. 12. 31. 제목개정 2014. 1. 1. 제54조의2공동사업에 대한 소득공제 등 특례 제51조의3 또는 조세특례제한법에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출납입투자출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출납입투자출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. 개정 2012. 1. 1., 2014. 1. 1. 전문개정 2009. 12. 31. 제목개정 2014. 1. 1. 제4절 세액의 계산 개정 2009. 12. 31. 제1관 세율 개정 2009. 12. 31. 제55조세율 거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액이하 종합소득산출세액이라 한다을 그 세액으로 한다. 개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31. | 종합소득 과세표준 | 세율 | |---|---| | 1,400만원 이하 | 과세표준의 6퍼센트 | | 1,400만원 초과 5,000만원 이하 | 84만원 1,400만원을 초과하는 금액의 15퍼센트 | | 5,000만원 초과 8,800만원 이하 | 624만원 5,000만원을 초과하는 금액의 24퍼센트 | | 8,800만원 초과 1억5천만원 이하 | 1,536만원 8,800만원을 초과하는 금액의 35퍼센트 | | 1억5천만원 초과 3억원 이하 | 3,706만원 1억5천만원을 초과하는 금액의 38퍼센트 | | 3억원 초과 5억원 이하 | 9,406만원 3억원을 초과하는 금액의 40퍼센트 | | 5억원 초과 10억원 이하 | 1억7,406만원 5억원을 초과하는 금액의 42퍼센트 | | 10억원 초과 | 

In [ ]:
# 제목을 추출하는 함수(사용 llm은 exaone3.5:2.4b) ollama list -> ollama pull exaone3.5:2.4b
from langchain_ollama import ChatOllama
def extract_title_with_llm(content):
    "exaone 3.5를 사용하여 content의 제목을 반환"
    titl_extractor_llm = ChatOllama(
        model = 'exaone3.5:2.4b',
        temperature=0.1
    )
    propmt = f"""다음 소득세법 조문의 핵심 제목을 100토큰 이내로 완벽하게 말이 되도록 추출해 주세요.
    중간에 말이 끊기면 안 되요.
    조문 : {content}"""